# RL Upper-Bound Focus8 Repro Notebook

Run all cells to reproduce the 8-knot improvements used to demonstrate better-than-paper behavior.

This notebook is intentionally configured for reproducibility and speed:

1. target override to the fixed Focus8 knot list
2. deterministic pre-pass enabled
3. RL stages disabled for this validation run
4. JSONL export written automatically to `outputs/paper_beater_focus8_results.jsonl`


In [1]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt


In [2]:

import os, re, json, ast, math, random, csv, glob, time
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


/home/ivan_sidorov/projects/upperbounds/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ivan_sidorov/projects/upperbounds/.venv/lib/python3.10/site-packages/plink/gui.py:33: UserWarning: Plink failed to import tkinter, GUI will not be available
  warnings.warn('Plink failed to import tkinter, GUI will not be available')


In [3]:
# Local repository paths (no Google Drive needed)
from pathlib import Path

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "data").exists() or (cand / "models").exists() or (cand / "training_data").exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
TRAINING_DIR = REPO_ROOT / "training_data"
OUT_DIR = REPO_ROOT / "outputs"

for folder in [DATA_DIR, MODELS_DIR, TRAINING_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_DIR / "unknotting.xlsx"
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Missing workbook: {XLSX_PATH}\n"
        "Place your database at repo-root/data/unknotting.xlsx"
    )

BASE = REPO_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("Workbook :", XLSX_PATH)
print("MODELS   :", MODELS_DIR)
print("TRAINING :", TRAINING_DIR)
print("OUTPUTS  :", OUT_DIR)


REPO_ROOT: /home/ivan_sidorov/projects/upperbounds
Workbook : /home/ivan_sidorov/projects/upperbounds/data/unknotting.xlsx
MODELS   : /home/ivan_sidorov/projects/upperbounds/models
TRAINING : /home/ivan_sidorov/projects/upperbounds/training_data
OUTPUTS  : /home/ivan_sidorov/projects/upperbounds/outputs


In [4]:
# ----------------------------
# Configuration
# ----------------------------
PROCESS_MODE = "bounds_eq"

TARGET_LOWER = 2
TARGET_UPPER = 3

START_INDEX = 0
END_INDEX = 300
FIRST_N = 300

# staged search (iterative deepening)
SEARCH_STAGES = [
    {"name": "SKIP", "num_variants": 0, "episodes_per_flip": 0, "max_steps": 1},
]
STOP_WHEN_EXACT = True

# multi-pass per knot (repeat staged search while upper bound keeps improving)
ENABLE_MULTI_PASS = False
MAX_KNOT_PASSES = 3

# two-flip exploration (sampled pairs per inflated variant)
ENABLE_TWO_FLIP = False
TWO_FLIP_MIN_STAGE_INDEX = 1      # 0-based index inside SEARCH_STAGES
MAX_TWO_FLIP_SAMPLES_PER_VARIANT = 40

# deterministic pre-pass before RL stages
ENABLE_DETERMINISTIC_PREPASS = True
PREPASS_MAX_SINGLE_FLIPS = None
PREPASS_RUN_GLOBAL_SIMPLIFY = True

BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20

# fallback defaults (used if SEARCH_STAGES is edited away)
NUM_VARIANTS_PER_KNOT = 12
UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500

TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# identification settings
USE_JONES = True
USE_HFK = False
PRECOMPUTE_HFK_FOR_ALL_ROWS = False
MAX_CROSSINGS_FOR_HFK = 13

# saving/output
SAVE_EVERY_KNOT = False
SAVE_AFTER_EACH_TARGET = False
WRITE_UPDATED_COPY = False
MAKE_TIMESTAMPED_BACKUP = False

# trust matching only in database crossing window
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

MAX_FLIPS_PER_VARIANT = None

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]


In [5]:
# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
_int_pat = re.compile(r'-?\d+')

def pick_first_existing(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_pd_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, list):
        return [[int(y) for y in q] for q in x]
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list) and all(isinstance(q, (list, tuple)) and len(q) == 4 for q in obj):
            return [[int(y) for y in q] for q in obj]
    except Exception:
        pass
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', s)
    if items:
        out = []
        for it in items:
            nums = [int(z.strip()) for z in it.split(',')]
            if len(nums) != 4:
                return None
            out.append(nums)
        return out
    return None

def parse_vector_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [int(z) for z in v]
        except Exception:
            pass
    nums = _int_pat.findall(s)
    if not nums:
        return None
    return [int(z) for z in nums]

def ensure_minmax_coeffs(v):
    if v is None:
        return None
    v = [int(x) for x in v]
    if len(v) < 3:
        return None
    mn, mx = v[0], v[1]
    coeffs = v[2:]
    if len(coeffs) != abs(mx - mn) + 1:
        return None
    return mn, mx, coeffs

def strip_leading_trailing_zeros(coeffs):
    coeffs = list(map(int, coeffs))
    i, j = 0, len(coeffs)
    while i < j and coeffs[i] == 0:
        i += 1
    while j > i and coeffs[j-1] == 0:
        j -= 1
    out = coeffs[i:j]
    return out if out else [0]

def canon_coeff_key(coeffs):
    return tuple(strip_leading_trailing_zeros(coeffs))

def canon_coeff_key_mirror(coeffs):
    return tuple(reversed(strip_leading_trailing_zeros(coeffs)))

def span_abs(mn, mx):
    return abs(int(mx) - int(mn))

def parse_unknotting_entry(x):
    """
    Returns dict with:
      kind: 'missing' | 'exact' | 'range' | 'other'
      lower, upper: ints or None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}
    if isinstance(x, (int, np.integer)):
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}
    if isinstance(x, float) and float(x).is_integer():
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}

    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) == 2:
            a, b = int(obj[0]), int(obj[1])
            if a == b:
                return {"kind": "exact", "lower": a, "upper": b, "raw": x}
            return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}
    except Exception:
        pass

    nums = [int(z) for z in _int_pat.findall(s)]
    if len(nums) == 1:
        return {"kind": "exact", "lower": nums[0], "upper": nums[0], "raw": x}
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        if a == b:
            return {"kind": "exact", "lower": a, "upper": b, "raw": x}
        return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}

    return {"kind": "other", "lower": None, "upper": None, "raw": x}

def format_unknotting(lower, upper):
    if lower is None and upper is None:
        return None
    if lower is None or upper is None:
        return None
    return str([int(lower), int(upper)])



def normalize_invariant_key_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, dict):
        return json.dumps(x, sort_keys=True, separators=(",", ":"))
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("{") and s.endswith("}"):
        try:
            obj = json.loads(s)
            return json.dumps(obj, sort_keys=True, separators=(",", ":"))
        except Exception:
            return s
    return s


In [6]:
# ----------------------------
# Load workbook and identify columns
# ----------------------------
df = pd.read_excel(XLSX_PATH)

knot_col  = pick_first_existing(df, ["knot_id", "name", "knot", "id"])
jones_col = pick_first_existing(df, ["jones_vector", "jones_polynomial_vector"])
pd_col    = pick_first_existing(df, ["pd_presentation", "pd_notation", "pd", "pd_code"])
u_col     = pick_first_existing(df, ["unknotting_number", "unknotting", "u"])
hfk_col   = pick_first_existing(df, ["hfk_invariant_key", "strong_invariant_key", "floer_invariant_key"])

print("Columns:")
print("  knot_col :", knot_col)
print("  jones_col:", jones_col)
print("  pd_col   :", pd_col)
print("  u_col    :", u_col)
print("  hfk_col  :", hfk_col)

if knot_col is None or pd_col is None or u_col is None:
    raise ValueError("Could not identify the required knot / PD / unknotting-number columns.")

if jones_col is None:
    jones_col = "jones_vector"
    df[jones_col] = None
    print(f"Created missing Jones column: {jones_col}")

if hfk_col is None:
    hfk_col = "hfk_invariant_key"
    df[hfk_col] = None
    print(f"Created missing HFK key column: {hfk_col}")

display(df.head())


Columns:
  knot_col : knot_id
  jones_col: jones_vector
  pd_col   : pd_presentation
  u_col    : unknotting_number
  hfk_col  : None
Created missing HFK key column: hfk_invariant_key


,knot_id,jones_vector,pd_presentation,unknotting_number,hfk_invariant_key
0,0_1,1,NaN,0,None
1,3_1,"[1, 4, 1, 0, 1, -1]","[[1,5,2,4],[3,1,4,6],[5,3,6,2]]",1,None
2,4_1,"[-2, 2, 1, -1, 1, -1, 1]","[[4,2,5,1],[8,6,1,5],[6,3,7,4],[2,7,3,8]]",1,None
3,5_1,"[2, 7, 1, 0, 1, -1, 1, -1]","[[2,8,3,7],[4,10,5,9],[6,2,7,1],[8,4,9,3],[10,...",2,None
4,5_2,"[1, 6, 1, -1, 2, -1, 1, -1]","[[1,5,2,4],[3,9,4,8],[5,1,6,10],[7,3,8,2],[9,7...",1,None


In [7]:
# ----------------------------
# Jones polynomial code from the attached notebook
# ----------------------------
def poly_add(p, q):
    r = dict(p)
    for e, c in q.items():
        r[e] = r.get(e, 0) + c
        if r[e] == 0:
            del r[e]
    return r

def poly_mul(p, q):
    r = {}
    for e1, c1 in p.items():
        for e2, c2 in q.items():
            e = e1 + e2
            r[e] = r.get(e, 0) + c1 * c2
    return {e:c for e,c in r.items() if c != 0}

def poly_monom(exp, coeff=1):
    return {int(exp): int(coeff)}

def poly_scale(p, s):
    return {e: c*s for e,c in p.items() if c*s != 0}

class DSU:
    def __init__(self):
        self.p = {}
    def find(self, x):
        if x not in self.p:
            self.p[x] = x
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra
    def n_components(self):
        return len({self.find(x) for x in self.p})

def bracket_from_pd(pd):
    n = len(pd)
    Delta = poly_add(poly_scale(poly_monom(2), -1), poly_scale(poly_monom(-2), -1))

    labels = set()
    for (a,b,c,d) in pd:
        labels.update([a,b,c,d])

    total = {}
    for mask in range(1 << n):
        dsu = DSU()
        for x in labels:
            dsu.find(x)

        a_count = 0
        b_count = 0
        for i, (a,b,c,d) in enumerate(pd):
            if ((mask >> i) & 1) == 0:
                a_count += 1
                dsu.union(a, b)
                dsu.union(c, d)
            else:
                b_count += 1
                dsu.union(b, c)
                dsu.union(d, a)

        loops = dsu.n_components()
        mon = poly_monom(a_count - b_count, 1)

        factor = {0: 1}
        for _ in range(loops - 1):
            factor = poly_mul(factor, Delta)

        total = poly_add(total, poly_mul(mon, factor))
    return total

def crossing_sign_pd(quad):
    a,b,c,d = quad
    return 1 if (a < c) == (b < d) else -1

def jones_string_from_pd(pd_list):
    if not pd_list:
        return None, "Empty PD"
    pdq = [tuple(map(int, q)) for q in pd_list]
    try:
        br = bracket_from_pd(pdq)
        w = sum(crossing_sign_pd(q) for q in pdq)
        sign = -1 if ((-3*w) % 2) else 1
        norm = poly_scale(poly_monom(-3*w, 1), sign)
        normed = poly_mul(norm, br)

        jt = {}
        for eA, c in normed.items():
            eT = Fraction(-eA, 4)
            jt[eT] = jt.get(eT, 0) + c
        jt = {e:c for e,c in jt.items() if c != 0}

        terms = []
        for e in sorted(jt.keys(), reverse=True):
            if e.denominator != 1:
                raise ValueError(f"Non-integral exponent encountered: {e}")
            c = jt[e]
            k = e.numerator
            if k == 0:
                mon = ""
            elif k == 1:
                mon = "t"
            else:
                mon = f"t^{k}"
            if mon == "":
                term = f"{c}"
            else:
                if c == 1:
                    term = mon
                elif c == -1:
                    term = "-" + mon
                else:
                    term = f"{c}*{mon}"
            terms.append(term)

        if not terms:
            return "0", None

        s = terms[0]
        for t in terms[1:]:
            if t.startswith("-"):
                s += " - " + t[1:]
            else:
                s += " + " + t
        return s, None
    except Exception as e:
        return None, repr(e)

def parse_jones_string_to_dict(jstr):
    if jstr is None:
        return None
    s = str(jstr).strip()
    if s == "" or s == "0":
        return {}

    s = s.replace(" - ", " + -")
    parts = [p.strip() for p in s.split(" + ") if p.strip()]
    poly = {}
    for term in parts:
        term = term.replace(" ", "")
        if "*t" in term:
            c_str, mon = term.split("*", 1)
            coeff = int(c_str)
        elif term.startswith("t") or term.startswith("-t"):
            coeff = -1 if term.startswith("-t") else 1
            mon = term[1:] if term.startswith("-t") else term
        else:
            coeff = int(term)
            exp = 0
            poly[exp] = poly.get(exp, 0) + coeff
            continue

        if mon == "t":
            exp = 1
        elif mon.startswith("t^"):
            exp = int(mon[2:])
        else:
            raise ValueError(f"Bad monomial format: {mon}")
        poly[exp] = poly.get(exp, 0) + coeff
    return {e:c for e,c in poly.items() if c != 0}

def poly_dict_to_knotinfo_vector(poly):
    if poly is None:
        return None
    if len(poly) == 0:
        return [0, 0, 0]
    mn, mx = min(poly.keys()), max(poly.keys())
    coeffs = [int(poly.get(e, 0)) for e in range(mn, mx + 1)]
    return [int(mn), int(mx)] + coeffs

def jones_vector_from_pd(pd_list):
    jstr, err = jones_string_from_pd(pd_list)
    if err is not None:
        return None, err
    poly = parse_jones_string_to_dict(jstr)
    vec = poly_dict_to_knotinfo_vector(poly)
    return vec, None



# ----------------------------
# Knot Floer key (mirror-aware)
# ----------------------------
def _as_int_or_none(x):
    if x is None:
        return None
    try:
        return int(x)
    except Exception:
        return None

def normalize_hfk_dict(hfk_obj):
    if not isinstance(hfk_obj, dict):
        return None

    out = {
        "L_space_knot": None if hfk_obj.get("L_space_knot") is None else bool(hfk_obj.get("L_space_knot")),
        "fibered": None if hfk_obj.get("fibered") is None else bool(hfk_obj.get("fibered")),
        "modulus": _as_int_or_none(hfk_obj.get("modulus")),
        "seifert_genus": _as_int_or_none(hfk_obj.get("seifert_genus")),
        "tau": _as_int_or_none(hfk_obj.get("tau")),
        "nu": _as_int_or_none(hfk_obj.get("nu")),
        "epsilon": _as_int_or_none(hfk_obj.get("epsilon")),
        "total_rank": _as_int_or_none(hfk_obj.get("total_rank")),
    }

    ranks = hfk_obj.get("ranks", {})
    norm_ranks = []
    if isinstance(ranks, dict):
        for k, v in ranks.items():
            try:
                a, m = k
                r = int(v)
            except Exception:
                continue
            norm_ranks.append([int(a), int(m), int(r)])
    norm_ranks.sort(key=lambda t: (t[0], t[1], t[2]))
    out["ranks"] = norm_ranks

    if out["total_rank"] is None and norm_ranks:
        out["total_rank"] = int(sum(t[2] for t in norm_ranks))

    return out

def hfk_key_from_link(L):
    hfk = L.knot_floer_homology()
    norm = normalize_hfk_dict(hfk)
    if norm is None:
        raise ValueError("knot_floer_homology returned non-dict output")
    return json.dumps(norm, sort_keys=True, separators=(",", ":"))

HFK_KEY_CACHE = {}

def hfk_keys_from_pd(pd_list, include_mirror=False):
    if not pd_list:
        return None, None, "Empty PD"

    pd_canon = json.dumps(pd_list, separators=(",", ":"))
    cache_key = ("mirror" if include_mirror else "direct", pd_canon)
    if cache_key in HFK_KEY_CACHE:
        k, km = HFK_KEY_CACHE[cache_key]
        return k, km, None

    try:
        L = Link(pd_list)
        k = hfk_key_from_link(L)
        km = None
        if include_mirror:
            km = hfk_key_from_link(L.mirror())

        HFK_KEY_CACHE[cache_key] = (k, km)
        if ("direct", pd_canon) not in HFK_KEY_CACHE:
            HFK_KEY_CACHE[("direct", pd_canon)] = (k, None)
        return k, km, None
    except Exception as e:
        return None, None, repr(e)


In [8]:
# ----------------------------
# Fill missing Jones vectors (and optionally HFK keys) in workbook memory
# ----------------------------
missing_before = 0
filled = 0
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling missing Jones vectors"):
    vec = parse_vector_cell(row.get(jones_col))
    if ensure_minmax_coeffs(vec) is not None:
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > 20:
        continue

    new_vec, err = jones_vector_from_pd(pd_list)
    if new_vec is not None:
        df.at[idx, jones_col] = str(new_vec)
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Jones entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])

if USE_HFK and PRECOMPUTE_HFK_FOR_ALL_ROWS:
    h_missing_before = 0
    h_filled = 0
    h_skipped = 0
    h_errors = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Precomputing HFK keys"):
        key = normalize_invariant_key_cell(row.get(hfk_col))
        if key is not None:
            df.at[idx, hfk_col] = key
            continue

        h_missing_before += 1
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is None:
            h_errors.append((idx, row.get(knot_col), "missing/bad PD"))
            continue
        if len(pd_list) > MAX_CROSSINGS_FOR_HFK:
            h_skipped += 1
            continue

        key, _, err = hfk_keys_from_pd(pd_list, include_mirror=False)
        if key is not None:
            df.at[idx, hfk_col] = key
            h_filled += 1
        else:
            h_errors.append((idx, row.get(knot_col), err))

    print("Missing HFK keys before:", h_missing_before)
    print("Filled HFK keys:", h_filled)
    print("Skipped by crossing cap:", h_skipped)
    print("HFK errors:", len(h_errors))


Filling missing Jones vectors: 100%|██████████| 12966/12966 [00:01<00:00, 9526.17it/s]

Missing Jones entries before: 1
Filled directly from PD: 0
Unfilled / errors: 1
Sample errors: [(0, '0_1', 'missing/bad PD')]


In [9]:
# ----------------------------
# Build lookup tables
# ----------------------------
lookup_jones = defaultdict(list)
lookup_hfk = defaultdict(list)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building hybrid lookups"):
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }

    if USE_JONES:
        vec = parse_vector_cell(row.get(jones_col))
        parsed = ensure_minmax_coeffs(vec)
        if parsed is not None:
            mn, mx, coeffs = parsed
            sp = span_abs(mn, mx)
            lookup_jones[(sp, canon_coeff_key(coeffs))].append({**rec, "mirror": False})
            lookup_jones[(sp, canon_coeff_key_mirror(coeffs))].append({**rec, "mirror": True})

    if USE_HFK:
        key = normalize_invariant_key_cell(row.get(hfk_col))
        mirror_key = None

        if key is None and PRECOMPUTE_HFK_FOR_ALL_ROWS:
            pd_list = parse_pd_cell(row.get(pd_col))
            if pd_list is not None and len(pd_list) <= MAX_CROSSINGS_FOR_HFK:
                key, mirror_key, err = hfk_keys_from_pd(pd_list, include_mirror=True)
                if key is not None:
                    df.at[idx, hfk_col] = key

        if key is not None:
            lookup_hfk[key].append({**rec, "mirror": False})
            if mirror_key is not None and mirror_key != key:
                lookup_hfk[mirror_key].append({**rec, "mirror": True})

print("Jones lookup keys:", len(lookup_jones))
print("HFK lookup keys  :", len(lookup_hfk))


Building hybrid lookups: 100%|██████████| 12966/12966 [00:02<00:00, 6271.21it/s]

Jones lookup keys: 16831
HFK lookup keys  : 0


In [10]:
# ----------------------------
# Training / RL utilities (v5-compatible semantics for inference)
# ----------------------------
import io
import gzip

MODEL_PATH_CANDIDATES = [
    MODELS_DIR / "best_model.zip",
    MODELS_DIR / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

LOCAL_EXTRA_FILES = [
    TRAINING_DIR / "hard_unknots.csv",
    TRAINING_DIR / "very_hard_unknots.csv",
    TRAINING_DIR / "random_diagrams.csv",
    TRAINING_DIR / "random_diagrams.txt",
    DATA_DIR / "hard_unknots.csv",
    DATA_DIR / "very_hard_unknots.csv",
]

def parse_link_strict(s: str):
    s = s.strip()
    try:
        obj = ast.literal_eval(s)
    except Exception as e:
        raise ValueError(f"Could not parse PD/DT string: {s[:80]}") from e
    if isinstance(obj, list):
        return Link(obj)
    raise ValueError("Parsed training example is not a list-like PD/DT object.")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)
            if crossings(L) == c0:
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    raw = []
    for _, row in df.iterrows():
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None:
            raw.append(json.dumps(pd_list))
            if max_keep is not None and len(raw) >= max_keep:
                break
    return raw

@dataclass
class EnvCfg:
    max_steps: int = UNKNOTTER_MAX_STEPS
    step_penalty: float = 0.05
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    w_delta: float = 1.0
    w_uphill: float = 0.5
    w_potential: float = 0.02

class SphKnotEnv(gym.Env):
    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)
        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        self.L = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        return min(3, self.num_actions - 1)

    def _obs(self):
        c = crossings(self.L)
        try:
            comps = len(self.L.link_components)
        except Exception:
            comps = 1
        tmp = Link(self.L.PD_code())
        try:
            reduced = tmp.simplify(mode="basic")
        except TypeError:
            reduced = tmp.simplify()
        can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
        recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
        return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))
        mode = self._map_blocked_mode(mode_req)

        c_before = crossings(self.L)
        if mode == 0:
            try:
                self.L.simplify(mode="basic")
            except TypeError:
                self.L.simplify()
        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)
        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)
        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        c_after = crossings(self.L)
        delta = c_before - c_after
        self._last_drop = max(delta, 0)

        reward = (
            self.cfg.w_delta * delta
            - self.cfg.w_uphill * max(0, -delta)
            - self.cfg.w_potential * c_after
            - self.cfg.step_penalty
        )

        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        if delta > 0:
            self._reset_blocks()
        else:
            if mode == 3:
                self._reset_blocks()
            elif delta < 0:
                self._blocked[mode] = True

        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

def load_training_pd_lines():
    raw = []

    for path in LOCAL_EXTRA_FILES:
        if path.exists():
            try:
                extra = read_first_col_local(str(path), has_header=True)
                raw += extra
                print(f"Loaded local extra {path.name}: {len(extra)}")
            except Exception as e:
                print(f"Could not load {path.name}: {e}")

    if not raw:
        raw = workbook_pd_lines()
        print(f"Falling back to workbook PD data: {len(raw)} examples")

    pd_lines = clean_pd_lines(raw, max_keep=None)
    random.Random(SEED).shuffle(pd_lines)
    if not pd_lines:
        raise RuntimeError(
            "No valid PD/DT strings available for training. "
            "Add local CSV/TXT files under training_data/ or ensure unknotting.xlsx contains PD data."
        )
    return pd_lines

def make_single_env(pd_list, cfg: EnvCfg):
    pd_str = json.dumps(pd_list)
    pd_lines_single = [pd_str]
    def _make():
        return SphKnotEnv(pd_lines_single, cfg)
    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg, episodes: int = 3, return_best_pd: bool = False):
    vec_env = make_single_env(pd_list, cfg)
    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(q) for q in pd_list]

    for _ in range(episodes):
        obs = vec_env.reset()
        best_crossings_ep = best_crossings_global
        best_pd_ep = best_pd_global

        for _step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)
            info = infos[0]
            cr = info.get("crossings", None)
            try:
                cur_pd = [list(q) for q in vec_env.envs[0].L.PD_code()]
            except Exception:
                cur_pd = None

            if cr is not None and cr < best_crossings_ep and cur_pd is not None:
                best_crossings_ep = cr
                best_pd_ep = cur_pd

            if cr == 0:
                success = True
                if cur_pd is not None:
                    best_crossings_ep = 0
                    best_pd_ep = cur_pd
                break
            if dones[0]:
                break

        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep
        if success:
            break

    vec_env.close()
    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global


In [11]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True)

# Robust fallback in case the config cell was edited or executed out of order.
if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = [
        BASE / "best_model.zip",
        BASE / "ppo_knot_rl_spherogram_continued.zip",
        OUT_DIR / "best_model.zip",
    ]

best_model_path = None
for path in MODEL_PATH_CANDIDATES:
    if Path(path).exists():
        best_model_path = Path(path)
        break

if best_model_path is not None:
    model = PPO.load(str(best_model_path), device="auto")
    print("Loaded existing model:", best_model_path)
else:
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            "No trained model found in MODEL_PATH_CANDIDATES, and TRAIN_IF_MODEL_MISSING=False."
        )
    print("No existing model found. Training a fresh one.")
    print("Searched these locations:")
    for p in MODEL_PATH_CANDIDATES:
        print("  -", p)

    pd_lines_train = load_training_pd_lines()
    vec_env = DummyVecEnv([lambda: SphKnotEnv(pd_lines_train, cfg)])
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=SEED,
        verbose=1,
    )
    model.learn(total_timesteps=TRAIN_STEPS_IF_NEEDED, progress_bar=True)
    best_model_path = OUT_DIR / "best_model.zip"
    model.save(str(best_model_path))
    vec_env.close()
    print("Saved trained model to:", best_model_path)

/home/ivan_sidorov/projects/upperbounds/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object learning_rate. Consider using `custom_objects` argument to replace this object.
Exception: code expected at most 16 arguments, got 18
  warnings.warn(
/home/ivan_sidorov/projects/upperbounds/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code expected at most 16 arguments, got 18
  warnings.warn(


Loaded existing model: /home/ivan_sidorov/projects/upperbounds/models/best_model.zip


In [12]:
# ----------------------------
# Inflation, flips, and matching helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_inflated_variants(pd_list,
                               num_variants=10,
                               backtrack_steps_min=6,
                               backtrack_steps_max=8,
                               riii_steps_max=20):
    variants = []
    L0 = Link(pd_list)

    for _ in range(num_variants):
        pd0 = L0.PD_code()
        if isinstance(pd0, list):
            L = Link(pd0)
        else:
            L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

        steps = random.randint(backtrack_steps_min, backtrack_steps_max)
        try:
            L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
        except Exception:
            pass
        try:
            L, _ = riii_shuffle_only_link(L, min(riii_steps_max, steps))
        except Exception:
            pass

        try:
            pd_new = [list(q) for q in L.PD_code()]
            variants.append(pd_new)
        except Exception:
            variants.append([list(q) for q in pd_list])

    if not variants:
        variants = [[list(q) for q in pd_list]]
    return variants

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append(((i,), flipped))
    return out

def generate_two_flip_variants_sampled(pd_list, max_samples=40):
    n = len(pd_list)
    if n < 2:
        return []

    all_pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    if max_samples is None or max_samples >= len(all_pairs):
        pairs = all_pairs
    else:
        pairs = random.sample(all_pairs, max_samples)

    out = []
    for i, j in pairs:
        flipped = []
        for k, quad in enumerate(pd_list):
            if k == i or k == j:
                flipped.append(flip_crossing_quad(quad))
            else:
                flipped.append(list(quad))
        out.append(((i, j), flipped))
    return out

def match_jones_vector_to_database(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return []
    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    return lookup_jones.get((sp, canon_coeff_key(coeffs)), [])

def match_hfk_key_to_database(key):
    if key is None:
        return []
    return lookup_hfk.get(key, [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)

def combine_invariant_matched_upper(j_matches, h_matches):
    """
    Returns (matched_upper, source_label, common_knots).
    Caller adds number of flips used.
    """
    if not j_matches and not h_matches:
        return None, None, []

    j_upper = best_upper_bound_from_matches(j_matches) if j_matches else None
    h_upper = best_upper_bound_from_matches(h_matches) if h_matches else None

    if j_upper is not None and h_upper is None:
        return int(j_upper), "jones", []
    if h_upper is not None and j_upper is None:
        return int(h_upper), "hfk", []

    j_map = defaultdict(list)
    for m in j_matches:
        if m.get("upper") is not None:
            j_map[str(m.get("knot"))].append(int(m["upper"]))

    h_map = defaultdict(list)
    for m in h_matches:
        if m.get("upper") is not None:
            h_map[str(m.get("knot"))].append(int(m["upper"]))

    common = sorted(set(j_map.keys()) & set(h_map.keys()))
    if common:
        common_upper = max(max(j_map[k] + h_map[k]) for k in common)
        return int(common_upper), "consensus", common

    return int(max(j_upper, h_upper)), "disagree_max", []

def identify_reduced_pd(best_pd):
    reduced_crossings = len(best_pd)
    if not (MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS):
        return None

    j_matches = []
    j_vec = None
    if USE_JONES:
        j_vec, j_err = jones_vector_from_pd(best_pd)
        if j_vec is not None:
            j_matches = match_jones_vector_to_database(j_vec)

    h_matches = []
    h_key = None
    if USE_HFK and reduced_crossings <= MAX_CROSSINGS_FOR_HFK:
        h_key, _, h_err = hfk_keys_from_pd(best_pd, include_mirror=False)
        if h_key is not None:
            h_matches = match_hfk_key_to_database(h_key)

    if not j_matches and not h_matches:
        return None

    matched_upper, source_label, common_knots = combine_invariant_matched_upper(j_matches, h_matches)
    if matched_upper is None:
        return None

    return {
        "matched_upper": int(matched_upper),
        "source": source_label,
        "common_knots": common_knots,
        "j_matches": j_matches,
        "h_matches": h_matches,
        "j_vec": j_vec,
        "h_key": h_key,
        "reduced_crossings": reduced_crossings,
    }


In [13]:
# ----------------------------
# Choose target rows
# ----------------------------
all_range_targets = []
for idx, row in df.iterrows():
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["kind"] == "range":
        all_range_targets.append({
            "row_index": int(idx),
            "knot": row.get(knot_col),
            "lower": int(uk["lower"]),
            "upper": int(uk["upper"]),
        })

print("All range-valued unknotting rows (all [a,b] with a != b):", len(all_range_targets))

bounds_targets = [
    t for t in all_range_targets
    if t["lower"] == TARGET_LOWER and t["upper"] == TARGET_UPPER
]
print(f"Rows with unknotting range [{TARGET_LOWER},{TARGET_UPPER}]:", len(bounds_targets))

neq_targets = list(all_range_targets)
print("Rows with non-exact unknotting range [a,b], a != b:", len(neq_targets))

if PROCESS_MODE == "all":
    targets = list(all_range_targets)
elif PROCESS_MODE == "first_n":
    targets = list(all_range_targets[:FIRST_N])
elif PROCESS_MODE == "slice":
    targets = list(all_range_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_eq":
    targets = list(bounds_targets)
elif PROCESS_MODE == "bounds_eq_slice":
    targets = list(bounds_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_neq":
    targets = list(neq_targets)
elif PROCESS_MODE == "bounds_neq_slice":
    targets = list(neq_targets[START_INDEX:END_INDEX])
else:
    raise ValueError(f"Unknown PROCESS_MODE: {PROCESS_MODE}")

print("Selected targets:", len(targets))
if len(targets) > 0:
    preview = pd.DataFrame(targets[:10])
    display(preview)
else:
    print("No targets selected.")


All range-valued unknotting rows (all [a,b] with a != b): 7595
Rows with unknotting range [2,3]: 1262
Rows with non-exact unknotting range [a,b], a != b: 7595
Selected targets: 1262


,row_index,knot,lower,upper
0,90,10_6,2,3
1,95,10_11,2,3
2,131,10_47,2,3
3,135,10_51,2,3
4,138,10_54,2,3
5,145,10_61,2,3
6,160,10_76,2,3
7,161,10_77,2,3
8,163,10_79,2,3
9,184,10_100,2,3


In [14]:
# Fixed target override for reproducible Focus8 validation
FOCUS8_KNOTS = [
    '12a_462', '12a_639', '12a_680',
    '12n_135', '12n_208', '12n_212', '12n_90',
    '13a_15',
]

target_map = {str(t['knot']): t for t in all_range_targets}
targets = [target_map[k] for k in FOCUS8_KNOTS if k in target_map]

missing = [k for k in FOCUS8_KNOTS if k not in target_map]
if missing:
    print('Warning: missing focus knots in target map:', missing)

ENABLE_DETERMINISTIC_PREPASS = True
PREPASS_MAX_SINGLE_FLIPS = None
PREPASS_RUN_GLOBAL_SIMPLIFY = True

ENABLE_MULTI_PASS = False
ENABLE_TWO_FLIP = False
SEARCH_STAGES = [{"name": "SKIP", "num_variants": 0, "episodes_per_flip": 0, "max_steps": 1}]
STOP_WHEN_EXACT = True

SAVE_EVERY_KNOT = False
SAVE_AFTER_EACH_TARGET = False
WRITE_UPDATED_COPY = False
MAKE_TIMESTAMPED_BACKUP = False

print('Focus targets selected:', len(targets))
print([t['knot'] for t in targets])


Focus targets selected: 8
['12a_462', '12a_639', '12a_680', '12n_135', '12n_208', '12n_212', '12n_90', '13a_15']


In [15]:
# ----------------------------
# Main improvement loop (supersearch)
# ----------------------------
if "SAVE_EVERY_KNOT" not in globals():
    SAVE_EVERY_KNOT = True
if "MIN_DATABASE_CROSSINGS" not in globals():
    MIN_DATABASE_CROSSINGS = 1
if "MAX_DATABASE_CROSSINGS" not in globals():
    MAX_DATABASE_CROSSINGS = 13

if "ENABLE_DETERMINISTIC_PREPASS" not in globals():
    ENABLE_DETERMINISTIC_PREPASS = True
if "PREPASS_MAX_SINGLE_FLIPS" not in globals():
    PREPASS_MAX_SINGLE_FLIPS = None
if "PREPASS_RUN_GLOBAL_SIMPLIFY" not in globals():
    PREPASS_RUN_GLOBAL_SIMPLIFY = True

if "SEARCH_STAGES" not in globals() or not SEARCH_STAGES:
    SEARCH_STAGES = [{
        "name": "fallback",
        "num_variants": NUM_VARIANTS_PER_KNOT,
        "episodes_per_flip": UNKNOTTER_EPISODES_PER_FLIP,
        "max_steps": UNKNOTTER_MAX_STEPS,
    }]

results_all = []

for tnum, target in enumerate(targets, start=1):
    idx = target["row_index"]
    knot_name = target["knot"]
    lower = int(target["lower"])
    upper_initial = int(target["upper"])

    print()
    print("=" * 80)
    print(f"[{tnum}/{len(targets)}] Processing:", knot_name, " current range:", [lower, upper_initial])

    original_pd = parse_pd_cell(df.at[idx, pd_col])
    if original_pd is None:
        print("  Skipping: bad or missing PD.")
        results_all.append({
            "row_index": idx,
            "knot": knot_name,
            "status": "bad_pd",
        })
        continue

    # ensure baseline identifiers when feasible
    base_vec = parse_vector_cell(df.at[idx, jones_col])
    if USE_JONES and ensure_minmax_coeffs(base_vec) is None and len(original_pd) <= 20:
        new_vec, err = jones_vector_from_pd(original_pd)
        if new_vec is not None:
            df.at[idx, jones_col] = str(new_vec)

    if USE_HFK and normalize_invariant_key_cell(df.at[idx, hfk_col]) is None and len(original_pd) <= MAX_CROSSINGS_FOR_HFK:
        key, _, err = hfk_keys_from_pd(original_pd, include_mirror=False)
        if key is not None:
            df.at[idx, hfk_col] = key

    working_upper = upper_initial
    best_evidence = None
    tried_total = 0
    matched_total = 0

    if ENABLE_DETERMINISTIC_PREPASS:
        print("  Deterministic pre-pass: single-flip reduction without RL")
        prepass_single = generate_single_flip_variants(original_pd)
        if PREPASS_MAX_SINGLE_FLIPS is not None:
            prepass_single = prepass_single[:int(PREPASS_MAX_SINGLE_FLIPS)]

        for flip_tuple, flipped_pd in prepass_single:
            tried_total += 1
            try:
                Lp = Link(flipped_pd)
            except Exception:
                continue

            if PREPASS_RUN_GLOBAL_SIMPLIFY:
                try:
                    Lp.simplify("global")
                except Exception:
                    try:
                        Lp.simplify()
                    except Exception:
                        pass
            else:
                try:
                    Lp.simplify()
                except Exception:
                    pass

            if len(Lp.crossings) == 0:
                matched_total += 1
                candidate_upper = 1
                if candidate_upper < working_upper:
                    working_upper = candidate_upper
                    best_evidence = {
                        "pass": "prepass",
                        "stage": "D1",
                        "variant_index": 0,
                        "flips": list(flip_tuple),
                        "flip_count": 1,
                        "candidate_upper": int(candidate_upper),
                        "source": "prepass_unknot",
                        "matched_upper": 0,
                        "common_knots": [],
                        "rl_success": False,
                        "min_crossings_found": 0,
                        "best_pd_crossings": 0,
                        "matched_knots_jones": [],
                        "matched_knots_hfk": [],
                        "best_pd": [],
                        "jones_vector": None,
                        "hfk_key": None,
                    }
                    print(f"    Improved in pre-pass via 1-flip {flip_tuple}: upper -> {working_upper}")
                if STOP_WHEN_EXACT and working_upper <= lower:
                    break
                continue

            try:
                pre_pd = [list(q) for q in Lp.PD_code()]
            except Exception:
                continue

            ident = identify_reduced_pd(pre_pd)
            if ident is None:
                continue

            matched_total += 1
            candidate_upper = ident["matched_upper"] + 1
            if candidate_upper < working_upper:
                working_upper = candidate_upper
                best_evidence = {
                    "pass": "prepass",
                    "stage": "D1",
                    "variant_index": 0,
                    "flips": list(flip_tuple),
                    "flip_count": 1,
                    "candidate_upper": int(candidate_upper),
                    "source": ident["source"],
                    "matched_upper": int(ident["matched_upper"]),
                    "common_knots": ident["common_knots"],
                    "rl_success": False,
                    "min_crossings_found": int(len(pre_pd)),
                    "best_pd_crossings": int(ident["reduced_crossings"]),
                    "matched_knots_jones": sorted(
                        {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["j_matches"]},
                        key=lambda x: (x[0], x[1], x[2])
                    ) if ident["j_matches"] else [],
                    "matched_knots_hfk": sorted(
                        {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["h_matches"]},
                        key=lambda x: (x[0], x[1], x[2])
                    ) if ident["h_matches"] else [],
                    "best_pd": pre_pd,
                    "jones_vector": ident["j_vec"],
                    "hfk_key": ident["h_key"],
                }
                print(f"    Improved in pre-pass via 1-flip {flip_tuple}: upper -> {working_upper}")

            if STOP_WHEN_EXACT and working_upper <= lower:
                break

    pass_limit = MAX_KNOT_PASSES if ENABLE_MULTI_PASS else 1

    for pidx in range(pass_limit):
        if STOP_WHEN_EXACT and working_upper <= lower:
            break

        pass_name = f"P{pidx+1}"
        print(f"  Starting pass {pass_name} with working upper {working_upper}")
        pass_improved = False

        for sidx, stage in enumerate(SEARCH_STAGES):
            if STOP_WHEN_EXACT and working_upper <= lower:
                break

            stage_name = stage.get("name", f"S{sidx+1}")
            stage_variants = int(stage.get("num_variants", NUM_VARIANTS_PER_KNOT))
            stage_eps = int(stage.get("episodes_per_flip", UNKNOTTER_EPISODES_PER_FLIP))
            stage_steps = int(stage.get("max_steps", UNKNOTTER_MAX_STEPS))
            cfg_stage = EnvCfg(max_steps=stage_steps, allow_backtrack=True)

            print(f"    Stage {stage_name}: variants={stage_variants}, episodes={stage_eps}, max_steps={stage_steps}")

            variants = generate_inflated_variants(
                original_pd,
                num_variants=stage_variants,
                backtrack_steps_min=BACKTRACK_STEPS_MIN,
                backtrack_steps_max=BACKTRACK_STEPS_MAX,
                riii_steps_max=RIII_STEPS_MAX,
            )

            stage_improved = False

            for vnum, variant_pd in enumerate(variants, start=1):
                if STOP_WHEN_EXACT and working_upper <= lower:
                    break

                # one-flip search
                one_flip_variants = generate_single_flip_variants(variant_pd)
                if MAX_FLIPS_PER_VARIANT is not None:
                    one_flip_variants = one_flip_variants[:int(MAX_FLIPS_PER_VARIANT)]

                for flip_tuple, flipped_pd in one_flip_variants:
                    tried_total += 1
                    success, min_crossings_found, best_pd = run_unknotter_on_pd(
                        flipped_pd,
                        model,
                        cfg_stage,
                        episodes=stage_eps,
                        return_best_pd=True,
                    )

                    if best_pd is None:
                        continue

                    ident = identify_reduced_pd(best_pd)
                    if ident is None:
                        continue

                    matched_total += 1
                    candidate_upper = ident["matched_upper"] + 1

                    if candidate_upper < working_upper:
                        working_upper = candidate_upper
                        stage_improved = True
                        pass_improved = True

                        best_evidence = {
                            "pass": pass_name,
                            "stage": stage_name,
                            "variant_index": vnum,
                            "flips": list(flip_tuple),
                            "flip_count": 1,
                            "candidate_upper": int(candidate_upper),
                            "source": ident["source"],
                            "matched_upper": int(ident["matched_upper"]),
                            "common_knots": ident["common_knots"],
                            "rl_success": bool(success),
                            "min_crossings_found": int(min_crossings_found),
                            "best_pd_crossings": int(ident["reduced_crossings"]),
                            "matched_knots_jones": sorted(
                                {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["j_matches"]},
                                key=lambda x: (x[0], x[1], x[2])
                            ) if ident["j_matches"] else [],
                            "matched_knots_hfk": sorted(
                                {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["h_matches"]},
                                key=lambda x: (x[0], x[1], x[2])
                            ) if ident["h_matches"] else [],
                            "best_pd": best_pd,
                            "jones_vector": ident["j_vec"],
                            "hfk_key": ident["h_key"],
                        }
                        print(f"      Improved via 1-flip {flip_tuple}: upper -> {working_upper}")

                        if STOP_WHEN_EXACT and working_upper <= lower:
                            break

                if STOP_WHEN_EXACT and working_upper <= lower:
                    break

                # sampled two-flip search (optional)
                if ENABLE_TWO_FLIP and sidx >= TWO_FLIP_MIN_STAGE_INDEX and not (STOP_WHEN_EXACT and working_upper <= lower):
                    two_flip_variants = generate_two_flip_variants_sampled(
                        variant_pd,
                        max_samples=MAX_TWO_FLIP_SAMPLES_PER_VARIANT,
                    )

                    for flip_tuple, flipped2_pd in two_flip_variants:
                        tried_total += 1
                        success, min_crossings_found, best_pd = run_unknotter_on_pd(
                            flipped2_pd,
                            model,
                            cfg_stage,
                            episodes=stage_eps,
                            return_best_pd=True,
                        )

                        if best_pd is None:
                            continue

                        ident = identify_reduced_pd(best_pd)
                        if ident is None:
                            continue

                        matched_total += 1
                        candidate_upper = ident["matched_upper"] + 2

                        if candidate_upper < working_upper:
                            working_upper = candidate_upper
                            stage_improved = True
                            pass_improved = True

                            best_evidence = {
                                "pass": pass_name,
                                "stage": stage_name,
                                "variant_index": vnum,
                                "flips": list(flip_tuple),
                                "flip_count": 2,
                                "candidate_upper": int(candidate_upper),
                                "source": ident["source"],
                                "matched_upper": int(ident["matched_upper"]),
                                "common_knots": ident["common_knots"],
                                "rl_success": bool(success),
                                "min_crossings_found": int(min_crossings_found),
                                "best_pd_crossings": int(ident["reduced_crossings"]),
                                "matched_knots_jones": sorted(
                                    {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["j_matches"]},
                                    key=lambda x: (x[0], x[1], x[2])
                                ) if ident["j_matches"] else [],
                                "matched_knots_hfk": sorted(
                                    {(str(m["knot"]), int(m["upper"]), bool(m.get("mirror", False))) for m in ident["h_matches"]},
                                    key=lambda x: (x[0], x[1], x[2])
                                ) if ident["h_matches"] else [],
                                "best_pd": best_pd,
                                "jones_vector": ident["j_vec"],
                                "hfk_key": ident["h_key"],
                            }
                            print(f"      Improved via 2-flip {flip_tuple}: upper -> {working_upper}")

                            if STOP_WHEN_EXACT and working_upper <= lower:
                                break

            if STOP_WHEN_EXACT and working_upper <= lower:
                print("    Exact interval reached; stopping remaining stages.")
                break

            if not stage_improved:
                print(f"    No improvement in stage {stage_name}.")

        if not pass_improved:
            print(f"  No improvement in pass {pass_name}; stopping passes.")
            break

    improved = working_upper < upper_initial
    if improved:
        df.at[idx, u_col] = format_unknotting(lower, working_upper)
        if SAVE_EVERY_KNOT:
            df.to_excel(XLSX_PATH, index=False)
            print("  Saved updated workbook to:", XLSX_PATH)
    else:
        print("  No improvement found.")

    result = {
        "row_index": idx,
        "knot": knot_name,
        "old_lower": lower,
        "old_upper": upper_initial,
        "new_upper": working_upper,
        "improved": improved,
        "tried_flip_runs": tried_total,
        "matched_flip_runs": matched_total,
        "evidence": best_evidence,
    }
    results_all.append(result)

    jsonl_path = OUT_DIR / "upper_bound_pass_results_supersearch.jsonl"
    with open(jsonl_path, "a") as f:
        f.write(json.dumps(result))
        f.write(chr(10))

print()
print("Done.")



[1/8] Processing: 12a_462  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (1,): upper -> 2

[2/8] Processing: 12a_639  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (1,): upper -> 2

[3/8] Processing: 12a_680  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (1,): upper -> 2

[4/8] Processing: 12n_135  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (8,): upper -> 2

[5/8] Processing: 12n_208  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (4,): upper -> 2

[6/8] Processing: 12n_212  current range: [2, 3]
  Deterministic pre-pass: single-flip reduction without RL
    Improved in pre-pass via 1-flip (10,): upper -> 2

[7/8] Processing: 12n_90  

Run this notebook with **Run All**. The final cell exports JSONL and prints a PASS/FAIL check for each focus knot.


In [16]:
# Export + verification
import json

out_path = OUT_DIR / 'paper_beater_focus8_results.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for r in results_all:
        rec = dict(r)
        if 'old_lower' in rec:
            rec.setdefault('new_lower', rec['old_lower'])
        rec.setdefault('run_tag', 'focus8_validation')
        f.write(json.dumps(rec, ensure_ascii=True))
        f.write('\n')

print('Wrote', out_path, 'rows:', len(results_all))

by_knot = {str(r.get('knot')): r for r in results_all}
print('\nVerification: expected [2,3] -> [2,2]')
all_ok = True
for k in FOCUS8_KNOTS:
    r = by_knot.get(k)
    if r is None:
        print(f'  {k}: MISSING')
        all_ok = False
        continue
    old_pair = [r.get('old_lower'), r.get('old_upper')]
    new_pair = [r.get('old_lower'), r.get('new_upper')]
    ok = (old_pair == [2, 3] and new_pair == [2, 2] and bool(r.get('improved')))
    label = 'PASS' if ok else 'FAIL'
    print(f'  {k}: {old_pair} -> {new_pair} | improved={r.get("improved")} | {label}')
    all_ok = all_ok and ok

print('\nOverall:', 'PASS' if all_ok else 'FAIL')


Wrote /home/ivan_sidorov/projects/upperbounds/outputs/paper_beater_focus8_results.jsonl rows: 8

Verification: expected [2,3] -> [2,2]
  12a_462: [2, 3] -> [2, 2] | improved=True | PASS
  12a_639: [2, 3] -> [2, 2] | improved=True | PASS
  12a_680: [2, 3] -> [2, 2] | improved=True | PASS
  12n_135: [2, 3] -> [2, 2] | improved=True | PASS
  12n_208: [2, 3] -> [2, 2] | improved=True | PASS
  12n_212: [2, 3] -> [2, 2] | improved=True | PASS
  12n_90: [2, 3] -> [2, 2] | improved=True | PASS
  13a_15: [2, 3] -> [2, 2] | improved=True | PASS

Overall: PASS
